# GRAM — Colab training run (step 8)
Runtime → Change runtime type → **GPU**. Everything persistent (code zip, data cache, runs) lives in `MyDrive/gram/`.

1. Upload `gram_code.zip` (made locally with `git archive -o dist/gram_code.zip HEAD`) to `MyDrive/gram/`.
2. Run all cells. If the session dies, re-run all cells: training resumes from the last checkpoint on Drive.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/gram'

In [ ]:
# Code: unzip the archive from Drive (or replace with `!git clone <your repo> /content/gram`).
!rm -rf /content/gram && mkdir -p /content/gram && unzip -q {WORK}/gram_code.zip -d /content/gram
%cd /content/gram
!pip -q install "tokenizers>=0.15"
!python -m pytest -q 2>&1 | tail -2

In [ ]:
# Data: build once (~15-20 min, downloads ~830 MB from Hugging Face), then reuse the cached tar from Drive
# so every run trains on byte-identical data.
import os
TAR = f'{WORK}/data_processed.tar'
if os.path.exists(TAR):
    !mkdir -p data && tar -xf {TAR} -C data
else:
    !python -m gram.data --out data/processed --raw /content/raw
    !tar -cf {TAR} -C data processed
!python -c "import json; m=json.load(open('data/processed/meta.json')); print('tokenizer', m['tokenizer_sha256'][:16], {d: sum(s['tokens'] for s in v.values()) for d, v in m['counts'].items()})"

In [ ]:
# Pick the run. Each RUN gets its own folder on Drive and resumes automatically.
#   main (plan §4):            RUN = 'gram_s0';       EXTRA = ''
#   full-access variant:       RUN = 'gram_xa50_s0';  EXTRA = '--p-xa 0.5'
#   seed-noise check:          RUN = 'gram_s1';       EXTRA = '--seed 1'
#   variant, core frozen:      RUN = 'gram_xa50fc_s0'; EXTRA = '--p-xa 0.5 --xa-freeze-core'
RUN = 'gram_xa50fc_s0'
EXTRA = '--p-xa 0.5 --xa-freeze-core'
OUT = f'{WORK}/runs/{RUN}'
CKPT = f'{OUT}/ckpt.pt'
if os.path.exists(CKPT):
    args = f'--resume {CKPT}'
else:
    args = f'--data-dir data/processed --out {OUT} --steps 10000 --batch-size 64 --eval-every 500 --seed 0 {EXTRA}'
!python -m gram.train {args}

## Go / no-go
Suppression results only mean something if the model actually learned Python and SQL **through its auxiliaries**.
Look for: full-profile Python/SQL loss far below the untrained ~9 nats, a clear rise when the module for that
domain is disabled, and little change in core loss.

In [ ]:
import json
rows = [json.loads(l) for l in open(f'{OUT}/eval.jsonl')]
r = rows[-1]; v = r['val']
print(f"step {r['step']}  tokens {r['tokens']}  {r['elapsed_s']/60:.1f} min")
print(f"{'profile':8s}" + ''.join(f'{d:>9s}' for d in ('core', 'python', 'sql')))
for p, row in v.items():
    print(f'{p:8s}' + ''.join(f'{row[d]:9.3f}' for d in ('core', 'python', 'sql')))
# Suppression is measured against each domain's own access profile; `full` is reported separately
# (both auxiliaries are never co-active in training, so full access is out-of-distribution).
print('ΔL python, Python module off (python -> sql profile):', round(v['sql']['python'] - v['python']['python'], 3))
print('ΔL sql,    SQL module off    (sql -> python profile):', round(v['python']['sql'] - v['sql']['sql'], 3))
print('ΔL core,   both off vs either one on:                ', round(v['core']['core'] - min(v['python']['core'], v['sql']['core']), 3))
print('full-access cost  python:', round(v['full']['python'] - v['python']['python'], 3), ' sql:', round(v['full']['sql'] - v['sql']['sql'], 3))


## Compare runs
Final validation matrix of every finished run, plus the two numbers the variant is meant to move:
the full-access cost (should shrink) and suppression when a module is off (should not shrink).

In [ ]:
import glob
for path in sorted(glob.glob(f'{WORK}/runs/*/eval.jsonl')):
    r = [json.loads(l) for l in open(path)][-1]; v = r['val']
    print(f"\n== {path.split('/')[-2]}  (step {r['step']})")
    for p, row in v.items():
        print(f'   {p:8s}' + ''.join(f'{row[d]:9.3f}' for d in ('core', 'python', 'sql')))
    print(f"   full-access cost  python {v['full']['python'] - v['python']['python']:+.3f}   sql {v['full']['sql'] - v['sql']['sql']:+.3f}")
    print(f"   module-off ΔL     python {v['sql']['python'] - v['python']['python']:+.3f}   sql {v['python']['sql'] - v['sql']['sql']:+.3f}")

## Held-out test evaluation
Scores every finished run on the **whole test split** (every profile × domain, with standard errors) and writes
`eval_test.json` next to each checkpoint plus one long-format CSV. Must run here: the harness refuses data whose
tokenizer differs from the one the checkpoint was trained with.

In [ ]:
!mkdir -p {WORK}/results && rm -f {WORK}/results/loss_test.csv
!python -m gram.evaluate {WORK}/runs/*/ckpt.pt --data-dir data/processed --batch-size 128 --csv {WORK}/results/loss_test.csv